In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['proyecto_bigdata']
coleccion = db['datos_scraping']

print("Conexión exitosa a MongoDB")

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

# --- PASO 1: CONFIGURACIÓN DEL DRIVER ---
options = Options()
options.add_argument("--headless")  # Ejecución invisible en Docker/servidor
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

# --- PASO 2: INICIALIZACIÓN DE VARIABLES ---
datos_totales = []          # Aquí se acumularán los datos de todas las páginas
paginas_objetivo = 5        # Número de páginas a recorrer
url_inicial = "https://books.toscrape.com/"  # URL inicial

driver.get(url_inicial)

try:
    # --- PASO 3: BUCLE DE PAGINACIÓN ---
    for i in range(paginas_objetivo):
        print(f"--- Procesando página {i + 1} ---")

        # Espera explícita para asegurar que los elementos carguen
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "product_pod"))
        )

        # CAPTURA: Extraer elementos de la vista actual
        productos = driver.find_elements(By.CLASS_NAME, "product_pod")
        for p in productos:
            datos_totales.append(p.text)

        # NAVEGACIÓN: Clic en el botón "Siguiente"
        try:
            boton_siguiente = driver.find_element(By.XPATH, "//li[@class='next']/a")
            boton_siguiente.click()
            time.sleep(3)  # Pausa para que cargue la nueva página
        except Exception:
            print("Se llegó a la última página o el botón no está disponible.")
            break

    print("Proceso finalizado con éxito.")
    print(f"Total de registros capturados: {len(datos_totales)}")
    print(datos_totales[:10])  # Muestra los primeros 10 resultados

except Exception as e:
    print(f"Error detectado durante la ejecución: {e}")

finally:
    # --- PASO 4: CIERRE SEGURO ---
    driver.quit()


Conexión exitosa a MongoDB
--- Procesando página 1 ---
--- Procesando página 2 ---
--- Procesando página 3 ---
--- Procesando página 4 ---
--- Procesando página 5 ---
Proceso finalizado con éxito.
Total de registros capturados: 100
['A Light in the ...\n£51.77\nIn stock\nAdd to basket', 'Tipping the Velvet\n£53.74\nIn stock\nAdd to basket', 'Soumission\n£50.10\nIn stock\nAdd to basket', 'Sharp Objects\n£47.82\nIn stock\nAdd to basket', 'Sapiens: A Brief History ...\n£54.23\nIn stock\nAdd to basket', 'The Requiem Red\n£22.65\nIn stock\nAdd to basket', 'The Dirty Little Secrets ...\n£33.34\nIn stock\nAdd to basket', 'The Coming Woman: A ...\n£17.93\nIn stock\nAdd to basket', 'The Boys in the ...\n£22.60\nIn stock\nAdd to basket', 'The Black Maria\n£52.15\nIn stock\nAdd to basket']
